# 04 — ClinicalBERT Fine-tuning
**Goal:** Fine-tune ClinicalBERT to beat baseline accuracy of 38%  
**Model:** medicalai/ClinicalBERT — pre-trained on 2M+ clinical notes (MIMIC-III)  
**Why ClinicalBERT over DistilBERT?** Domain-specific pre-training on medical text  
**Final Result:** 68.72% accuracy vs 38% baseline

In [ ]:
# Imports
import pandas as pd
import numpy as np
import pickle
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler
)
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# Load Data

df = pd.read_csv('mtsamples_cleaned.csv')
df = df.dropna(subset=['transcription'])

with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

print("Data loaded:", df.shape)
print("Specialties:", df['medical_specialty'].value_counts().to_dict())

In [ ]:
# Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    df['transcription'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Classes — Train: {len(y_train.unique())} | Test: {len(y_test.unique())}")

In [ ]:
# Dataset Class

class MedicalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
# Load Tokenizer & Create DataLoaders
tokenizer = AutoTokenizer.from_pretrained('medicalai/ClinicalBERT')

train_dataset = MedicalDataset(X_train, y_train, tokenizer)
test_dataset = MedicalDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches : {len(test_loader)}")

In [ ]:
# Load Model

model = AutoModelForSequenceClassification.from_pretrained(
    'medicalai/ClinicalBERT',
    num_labels=8
)

model = model.to(device)

print("ClinicalBERT loaded on:", device)
print("Number of labels:", 8)

In [ ]:
# Training & Evaluation Functions

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    loop = tqdm(loader, desc='Training')

    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item()

        loop.set_postfix(loss=loss.item(), acc=correct/total)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(
        all_labels, all_preds,
        target_names=le.classes_,
        zero_division=0
    )
    return accuracy, report, all_preds, all_labels

In [ ]:
# Training Loop

optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 4
num_training_steps = num_epochs * len(train_loader)

scheduler = get_scheduler(
    'linear',
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

best_accuracy = 0

for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")

    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, scheduler, device
    )

    val_accuracy, val_report, _, _ = evaluate(model, test_loader, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Accuracy: {val_accuracy:.4f}")

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        torch.save(model.state_dict(), 'best_model.pt')
        print(f"Best model saved — Accuracy: {best_accuracy:.4f}")

print(f"\nBaseline: 38% → ClinicalBERT Best: {best_accuracy*100:.2f}%")

In [ ]:
# Final Evaluation
# Load best model

model.load_state_dict(torch.load('best_model.pt'))
final_accuracy, final_report, all_preds, all_labels = evaluate(
    model, test_loader, device
)

print(f"Final Accuracy: {final_accuracy*100:.2f}%")
print("\nClassification Report:")
print(final_report)

In [ ]:
# Confusion Matrix

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    all_labels, all_preds,
    display_labels=le.classes_,
    xticks_rotation=45,
    ax=ax,
    cmap='Blues'
)

plt.title('ClinicalBERT — Confusion Matrix')

plt.tight_layout()

plt.savefig('clinicalbert_confusion_matrix.png', dpi=150)

plt.show()

In [ ]:
# Save Results

with open('results/clinicalbert_report.txt', 'w') as f:
    f.write("ClinicalBERT Fine-tuned Model\n")
    f.write("=" * 50 + "\n")
    f.write(f"Final Accuracy: {final_accuracy*100:.2f}%\n\n")
    f.write(final_report)

with open('results/model_comparison.txt', 'w') as f:
    f.write("Medical Specialty Classification — Model Comparison\n")
    f.write("=" * 50 + "\n")
    f.write(f"Baseline (TF-IDF + LR) : 38.00%\n")
    f.write(f"DistilBERT             : 63.55%\n")
    f.write(f"ClinicalBERT           : {final_accuracy*100:.2f}%\n")

print("All results saved!")

In [ ]:
# Test Inference
def predict_specialty(text):
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding='max_length',
        return_tensors='pt'
    )

    with torch.no_grad():
        outputs = model(
            input_ids=encoding['input_ids'].to(device),
            attention_mask=encoding['attention_mask'].to(device)
        )
        probs = torch.softmax(outputs.logits, dim=1)[0]
        pred_label = torch.argmax(probs).item()
        confidence = probs[pred_label].item()

    specialty = le.inverse_transform([pred_label])[0]
    top3_probs, top3_indices = torch.topk(probs, 3)
    top3 = [
        (le.inverse_transform([i.item()])[0], round(p.item()*100, 2))
        for i, p in zip(top3_indices, top3_probs)
    ]

    return specialty, round(confidence*100, 2), top3

In [ ]:
# Test sample

test_note = """
Patient presents with crushing substernal chest pain radiating to the left arm.
ECG shows ST elevation in V1-V4. Troponin elevated. Diagnosed with STEMI.
Dual antiplatelet therapy initiated. Urgent PCI planned.
"""

specialty, confidence, top3 = predict_specialty(test_note)

print(f"Predicted: {specialty} ({confidence}% confidence)")
print("Top 3:")
for rank, (spec, prob) in enumerate(top3, 1):
    print(f"  {rank}. {spec} — {prob}%")